# Fruits Vector Demo — Stage 1: Manual Embeddings
This stage walks through the idea and process of manually defining fruit vectors, persisting them into LanceDB, and querying the space even though mango itself is not stored.

## Imports and configuration
We only need `numpy`, `lancedb`, and `lancedb.pydantic` so the setup stays lightweight. The following code declares the local database path and table name that both stages will share.

In [ ]:
import numpy as np
import lancedb
from lancedb.pydantic import LanceModel, Vector
from pathlib import Path

DB_PATH = Path("fruits_lancedb")
TABLE_NAME = "fruit_vectors"

DB_PATH.mkdir(exist_ok=True)

## Schema and persistence helpers
This cell defines a `FruitVector` schema and a helper that recreates the LanceDB table for a clean run.

In [ ]:
db = lancedb.connect(str(DB_PATH))

class FruitVector(LanceModel):
    name: str
    vector: Vector(3)

def prepare_table():
    if TABLE_NAME in db.table_names():
        db.drop_table(TABLE_NAME)
    return db.create_table(TABLE_NAME, schema=FruitVector)

table = prepare_table()

## Insert handcrafted fruit vectors
No embedding model is invoked: we craft three-dimensional vectors for each fruit, insert them into LanceDB, and log the stored rows for transparency.

In [ ]:
fruit_vectors = {
    "papaya": np.array([0.88, 0.35, 0.05]),
    "peach": np.array([0.81, 0.32, 0.12]),
    "pear": np.array([0.62, 0.51, 0.38]),
    "pineapple": np.array([0.15, 0.85, 0.79]),
    "apple": np.array([0.28, 0.42, 0.69]),
    "apricot": np.array([0.77, 0.44, 0.2]),
    "guava": np.array([0.1, 0.74, 0.58]),
    "nectarine": np.array([0.83, 0.3, 0.1])
}
records = [
    {
        "name": name,
        "vector": vector.tolist(),
    }
    for name, vector in fruit_vectors.items()
]
table.add(records)
stored = table.to_pandas()
print(f"✅ Stored {len(stored)} handcrafted fruit vectors.")
print(stored[["name", "vector"]])

## Mango-like vector context + stage closeout
We now define the mango-like vector for context, but this final cell simply displays every stored LanceDB row so learners can inspect all handcrafted vectors and confirm mango is absent before continuing to Stage 2.

In [ ]:
mango_context = np.array([0.9, 0.34, 0.08])
stored = table.to_pandas()
print("✅ Stored handcrafted fruit vectors:")
print(stored[["name", "vector"]])
print("Available fruits:", ", ".join(stored['name'].tolist()))
print("✅ Stage 1 complete: the vector store is ready for Stage 2.")